# Artificial Intelligence Technology and Application

## Machine Learning Lab Guide - Student Version

Independent implementation prepared by **Sundetkhan Bekzat**.


# 1 Emotion Recognition of Customer Evaluations

This notebook keeps the lab objective but uses compact local examples so it can run without external datasets.


## 1.1 Text Cleaning
A small review set is normalized before vectorization.


In [1]:
import re
import pandas as pd

reviews = pd.DataFrame({
    "text": [
        "Fast delivery and excellent packaging!",
        "The item broke after one day",
        "Friendly support and good quality",
        "Late delivery, damaged box",
        "Great value for the price",
        "Bad support and poor instruction",
        "I love the camera quality",
        "The battery failed quickly",
        "Clean design and stable work",
        "Terrible smell from the package",
        "Helpful seller, smooth refund",
        "Worst cable I have bought",
    ],
    "positive": [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0],
})

def clean_text(value):
    return re.sub(r"[^a-z ]+", " ", value.lower()).strip()

reviews["clean"] = reviews["text"].map(clean_text)
print(reviews[["clean", "positive"]].head())


                                   clean  positive
0  fast delivery and excellent packaging         1
1           the item broke after one day         0
2      friendly support and good quality         1
3             late delivery  damaged box         0
4              great value for the price         1


## 1.2 Vectorization and Models
Naive Bayes and logistic regression are compared on TF-IDF features.


In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB, BernoulliNB

X_train, X_test, y_train, y_test = train_test_split(
    reviews["clean"], reviews["positive"], stratify=reviews["positive"], random_state=9, test_size=0.33
)
vectorizer = CountVectorizer(stop_words="english")
transformer = TfidfTransformer()
train_counts = vectorizer.fit_transform(X_train)
train_tfidf = transformer.fit_transform(train_counts)
test_tfidf = transformer.transform(vectorizer.transform(X_test))

models = {
    "multinomial_nb": MultinomialNB(),
    "bernoulli_nb": BernoulliNB(),
    "logistic": LogisticRegression(max_iter=500),
}
for name, estimator in models.items():
    estimator.fit(train_tfidf, y_train)
    print(name, round(accuracy_score(y_test, estimator.predict(test_tfidf)), 3))


multinomial_nb 0.75
bernoulli_nb 0.5
logistic 0.75


## 1.3 Inference
A new customer sentence is classified by the logistic model.


In [3]:
logistic = models["logistic"]
new_review = [clean_text("excellent support and fast refund")]
probability = logistic.predict_proba(transformer.transform(vectorizer.transform(new_review)))[0, 1]
print("positive probability:", round(probability, 3))


positive probability: 0.554
